# Data Preparation
This version of the data preparation is before having access to the full text of the articles. Instead, we're running a test using just article titles.

In [3]:
import pandas as pd
import numpy as np

## 0. Read and filter data

In [2]:
raw_df = pd.read_csv('../data/posts-export-by-page-views-Feb-01-2025-Mar-05-2026-Masthead-Maine.csv')
raw_df.head()

,Apikey,URL,Title,Publish date,Authors,Section,Tags,Sort (Views),Visitors,Views,...,Channel vis.,Website views,AMP views,Fb instant views,Post id,Views source,Views syndicated,Views by Site,High-Level Smart Tags,Low-Level Smart Tags
0,pressherald.com,https://www.pressherald.com,The Portland Press Herald,NaN,Staff,Uncategorized,NaN,13604769,1188026,13604769,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,centralmaine.com,https://www.centralmaine.com,Kennebec Journal and Morning Sentinel,NaN,Staff,Uncategorized,NaN,4571551,453052,4571551,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Law, Gov’t & Politics,Legal Issues",NaN
2,pressherald.com,http://www.pressherald.com/obituaries/,Obituaries,NaN,Staff,Uncategorized,NaN,3465736,410059,3465736,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,sunjournal.com,https://www.sunjournal.com,Lewiston Sun Journal,NaN,Staff,Uncategorized,NaN,3436858,367361,3436858,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,centralmaine.com,https://www.centralmaine.com/obituaries/,Obituaries,NaN,Staff,Uncategorized,NaN,3284383,275492,3284383,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
raw_df.shape

(10000, 47)

In [4]:
raw_df.columns

Index(['Apikey', 'URL', 'Title', 'Publish date', 'Authors', 'Section', 'Tags',
       'Sort (Views)', 'Visitors', 'Views', 'Avg. views', 'Engaged minutes',
       'Avg. minutes', 'New vis.', 'Views new vis.', 'Avg. views new vis.',
       'Minutes New Vis.', 'Avg. minutes new vis.', 'Returning vis.',
       'Views ret. vis.', 'Avg. views ret. vis.', 'Minutes Ret. Vis.',
       'Avg. minutes ret. vis.', 'Desktop views', 'Mobile views',
       'Tablet views', 'Search refs', 'Internal refs', 'Other refs',
       'Direct refs', 'Social refs', 'Fb refs', 'Tw refs', 'Pi refs',
       'Social interactions', 'Fb interactions', 'Pi interactions',
       'Channel vis.', 'Website views', 'AMP views', 'Fb instant views',
       'Post id', 'Views source', 'Views syndicated', 'Views by Site',
       'High-Level Smart Tags', 'Low-Level Smart Tags'],
      dtype='str')

In [5]:
raw_df['Post id'].isna().sum()

np.int64(883)

In [6]:
raw_df['Publish date'].isna().sum()

np.int64(883)

In [7]:
i = 500
print(raw_df.URL[i],'\n',raw_df['Post id'][i])

https://www.pressherald.com/2025/08/11/mother-accused-of-killing-3-year-old-in-milford-appears-in-court/ 
 https://www.pressherald.com/2025/08/11/mother-accused-of-killing-3-year-old-in-milford-appears-in-court/


I can either use 'URL' or 'Post id' as identifiable index. Can filter by either 'Post id' or 'Publish date'

In [8]:
art_df = raw_df[~raw_df['Post id'].isna()].reset_index(drop=True) # full "raw" dataset that only contains articles
art_df.shape

(9117, 47)

In [9]:
art_df['Publish date']

0       2026-01-09 08:55
1       2025-03-06 17:14
2       2026-01-28 17:43
3       2025-02-05 16:21
4       2025-09-01 04:00
              ...       
9112    2025-06-09 18:23
9113    2025-06-18 20:07
9114    2025-11-04 04:00
9115    2015-08-07 16:45
9116    2023-07-14 09:10
Name: Publish date, Length: 9117, dtype: str

In [ ]:
# check for articles older than Feb 01 2025
# first, convert dates from str to datetime objects
import datetime
art_df['Publish_date'] = art_df['Publish date'].apply(lambda x : datetime.datetime.strptime(x, '%Y-%m-%d %H:%M'))

print((art_df.Publish_date > datetime.datetime.strptime('2025-01-31 23:59:00', '%Y-%m-%d %H:%M:%S')).sum(),' of ', art_df.shape[0],' articles within date range.')


8731  of  9117  articles within date range.


In [11]:
# filter them out
art_df = art_df[art_df.Publish_date > datetime.datetime.strptime('2025-01-31 23:59:00', '%Y-%m-%d %H:%M:%S')].reset_index(drop=True)
art_df.shape

(8731, 48)

In [12]:
# load scraped text data
tagged_text = pd.read_csv('../data/text-tagged-articles.csv')
untagged_text = pd.read_csv('../data/text-untagged-articles.csv')
text_df = pd.concat([tagged_text,untagged_text],ignore_index=True)
print(text_df.shape)
text_df.head()

(8961, 2)


,url,text
0,http://www.centralmaine.com/2025/09/03/capitol...,Moments before the chief of the Maine Capitol ...
1,http://www.centralmaine.com/2025/09/03/childre...,"Michael Dowling, a pediatric dentist in Yarmou..."
2,http://www.centralmaine.com/2025/09/03/five-fo...,Week 1 of the Maine high school football seaso...
3,http://www.centralmaine.com/2025/09/03/hollywo...,"On Aug. 19,The New York Times reportedthat She..."
4,http://www.centralmaine.com/2025/09/03/maine-b...,Dan Kleban at Maine Beer Company on Tuesday. K...


In [13]:
art_df = art_df.merge(text_df, how='left', left_on='URL', right_on='url')
print(art_df.shape)
print(art_df.text[0])

(8731, 50)
Federal and local law enforcement are investigating alleged financial mismanagement in Gray on the heels of therecent resignation of the town manager. The investigation was first brought to light at a Town Council meeting on Tuesday, when Chairman Michael Johnson said that in November 2024, town officials discovered that Gray had spent approximately $1.25 million on a fire truck, but the town didn’t have the money. The fire truck purchase had been voted on in 2019, but wasn’t completed. Former Town Manager Michael Foley said he believed there was an additional $1.5 million available following a 2019 audit, and the truck was commissioned and paid for in February of 2025. However, in June, it was discovered that the funds had already been spent, which the board was not informed of at the time. About the same time, the town discovered another purchase of more than $140,000 in ambulance write-offs that had not been disclosed. In November, Foley stepped down from the role of town

## 1. 'User_Needs' column

In [14]:
art_df['Tag_list'] = art_df.Tags.apply(lambda x : x.split(','))

user_tags = []
for i in range(len(art_df)):
    flag = 0
    for tag in art_df['Tag_list'][i]:
        if 'user_need' in tag:
            user_tags.append(tag[len('user_need: '):])
            flag = 1
    if flag:
        continue
    user_tags.append('none') # this is how they're labeled originally

In [15]:
len(user_tags) # just checking that the len matches 

8731

In [16]:
art_df['User_Needs'] = user_tags
art_df.User_Needs.unique()

<ArrowStringArray>
[          'update-me',                'none', 'give-me-perspective',
          'educate-me',          'connect-me',             'help-me',
          'inspire-me',      'other-not-news']
Length: 8, dtype: str

NOTE: 'other-not-news' tag is NOT part of the User Needs tags. **Should it be treated as 'none'?**
--> We'll remove them!

In [17]:
art_df = art_df[(art_df.User_Needs != 'other-not-news')].reset_index(drop=True)
len(art_df)

8726

## 2. Clean data for EDA

(copy-pasted the list of columns for reference)<br />
       'Apikey', 'URL', 'Title', 'Publish date', 'Authors', 'Section', 'Tags',<br />
       'Sort (Views)', 'Visitors', 'Views', 'Avg. views', 'Engaged minutes',<br />
       'Avg. minutes', 'New vis.', 'Views new vis.', 'Avg. views new vis.',<br />
       'Minutes New Vis.', 'Avg. minutes new vis.', 'Returning vis.',<br />
       'Views ret. vis.', 'Avg. views ret. vis.', 'Minutes Ret. Vis.',<br />
       'Avg. minutes ret. vis.', 'Desktop views', 'Mobile views',<br />
       'Tablet views', 'Search refs', 'Internal refs', 'Other refs',<br />
       'Direct refs', 'Social refs', 'Fb refs', 'Tw refs', 'Pi refs',<br />
       'Social interactions', 'Fb interactions', 'Pi interactions',<br />
       'Channel vis.', 'Website views', 'AMP views', 'Fb instant views',<br />
       'Post id', 'Views source', 'Views syndicated', 'Views by Site'<br />

In [18]:
# choose columns to keep
columns = ['Apikey','URL','Title','text','Publish_date','Authors','Section','User_Needs', # metadata
           'Views','Avg. views','Engaged minutes','Avg. minutes', # some metrics (can be changed to include other metrics!)
           'Desktop views','Mobile views', 'Tablet views'] # device metrics
eda_df = art_df[columns]
eda_df.head()

,Apikey,URL,Title,text,Publish_date,Authors,Section,User_Needs,Views,Avg. views,Engaged minutes,Avg. minutes,Desktop views,Mobile views,Tablet views
0,"pressherald.com, sunjournal.com",http://www.pressherald.com/2026/01/09/gray-inv...,Gray investigated for buying $1.25M fire truck...,Federal and local law enforcement are investig...,2026-01-09 08:55:00,Rory Sweeting,News,update-me,109875,1.084,31960.0,0.315,3910.0,104307.0,1658.0
1,"centralmaine.com, pressherald.com, sunjournal.com",https://www.pressherald.com/2025/03/06/social-...,Social Security now requires Maine parents to ...,"An update, in which the Social Security Admini...",2025-03-06 17:14:00,Joe Lawlor,News,none,98329,1.109,64495.0,0.727,9783.0,86241.0,2305.0
2,"centralmaine.com, pressherald.com, sunjournal.com",http://www.pressherald.com/2026/01/28/ice-agen...,"ICE agents shatter window, leave 1-month-old b...","PORTLAND — Hassane Barry and his wife, Nene, b...",2026-01-28 17:43:00,"Dylan Tusinski,Salomé Cloteaux",News,give-me-perspective,76168,1.145,84939.0,1.277,12634.0,61819.0,1715.0
3,"centralmaine.com, pressherald.com, sunjournal.com",https://www.pressherald.com/2025/02/05/maine-m...,Maine Mall shooting: Police search for suspect...,SOUTH PORTLAND — Police are searching for a pe...,2025-02-05 16:21:00,"Daniel Kool,Morgan Womack",News,none,73901,1.341,38065.0,0.691,14806.0,58483.0,612.0
4,"centralmaine.com, pressherald.com, sunjournal.com",https://www.pressherald.com/2025/09/01/bob-dyl...,Bob Dylan and Willie Nelson headline Outlaw Mu...,NaN,2025-09-01 04:00:00,Aimsel Ponti,Life & Culture,none,64763,1.155,23763.0,0.424,2635.0,59514.0,2614.0


In [19]:
eda_df.count()

Apikey             8726
URL                8726
Title              8726
text               8606
Publish_date       8726
Authors            8726
Section            8726
User_Needs         8726
Views              8726
Avg. views         8726
Engaged minutes    8726
Avg. minutes       8726
Desktop views      8726
Mobile views       8726
Tablet views       8725
dtype: int64

In [20]:
eda_df['Tablet views'] = eda_df['Tablet views'].fillna(0)
eda_df.count()

Apikey             8726
URL                8726
Title              8726
text               8606
Publish_date       8726
Authors            8726
Section            8726
User_Needs         8726
Views              8726
Avg. views         8726
Engaged minutes    8726
Avg. minutes       8726
Desktop views      8726
Mobile views       8726
Tablet views       8726
dtype: int64

In [21]:
eda_df = eda_df.dropna(ignore_index=True)
eda_df.count()

Apikey             8606
URL                8606
Title              8606
text               8606
Publish_date       8606
Authors            8606
Section            8606
User_Needs         8606
Views              8606
Avg. views         8606
Engaged minutes    8606
Avg. minutes       8606
Desktop views      8606
Mobile views       8606
Tablet views       8606
dtype: int64

In [22]:
eda_df.to_csv('../data/EDA_data-FULL.csv', index=False) # save!

## 3. Clean data for ML

In [23]:
# we're just cutting out unneeded columns and splitting into tagged/untagged datasets
valid_tags = ['update-me','give-me-perspective','educate-me',
              'connect-me','help-me','inspire-me']
ml_cols = ['URL','Title','text','User_Needs']

tagged_df = eda_df[ml_cols][eda_df.User_Needs.isin(valid_tags)]
print(tagged_df.shape[0],' tagged.')

untagged_df = eda_df[ml_cols][~eda_df.User_Needs.isin(valid_tags)]
print(untagged_df.shape[0],' untagged.')

3855  tagged.
4751  untagged.


In [24]:
tagged_df.to_csv('../data/ML_tagged_data-FULL.csv',index=False)
untagged_df.to_csv('../data/ML_untagged_data-FULL.csv',index=False)